### Model Trainer

In [33]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# load dataset
df = pd.read_csv("data/clean_car_data.csv")

In [38]:
df['selling_price'] = np.log1p(df['selling_price'])

In [39]:
X = df.drop(columns=["selling_price"])

y = df["selling_price"]

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [41]:
numerical_columns = [
    "km_driven",
    "car_age"
]

categorical_columns = [
    "fuel",
    "seller_type",
    "transmission",
    "owner",
    "brand"
]

In [42]:
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [43]:
preprocessor = ColumnTransformer(
    [
        ("num", num_pipeline, numerical_columns),
        ("cat", cat_pipeline, categorical_columns)
    ]
)

In [44]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [45]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "XGBoost": XGBRegressor(),
    "CatBoost": CatBoostRegressor(verbose=False)
}

In [46]:
results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    r2 = r2_score(y_test, predictions)

    results[name] = r2

    print(f"{name} : {r2:.4f}")

Linear Regression : 0.7766
Decision Tree : 0.6124
Random Forest : 0.7552
XGBoost : 0.7765
CatBoost : 0.8078


In [48]:
results_df = pd.DataFrame(
    results.items(),
    columns=["Model", "R2 Score"]
)

results_df.sort_values(
    by="R2 Score",
    ascending=False
)

,Model,R2 Score
4,CatBoost,0.807834
0,Linear Regression,0.776620
3,XGBoost,0.776508
2,Random Forest,0.755150
1,Decision Tree,0.612387


In [55]:
best_model_name = max(
    results,
    key=results.get
)

best_model_score = results[best_model_name]

print(best_model_name)
print(best_model_score)

best_model = models[best_model_name]


CatBoost
0.80783376802535


In [56]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

pred = best_model.predict(X_test)

print("R2 :", r2_score(y_test, pred))
print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, pred)))

R2 : 0.80783376802535
MAE : 0.2797444424563141
RMSE : 0.3630031957952003
